In [ ]:
import pandas as pd

df = pd.read_json('https://raw.githubusercontent.com/Fireblend/squirdle/refs/heads/main/data/pokedex.json')
names = df.columns
df = df.transpose()
df.insert(0, 'Name', names)
df.columns = ['Name', 'Generation', 'Type1', 'Type2', 'Height', 'Weight']

df['Height'] = df['Height'].astype(float).round(2)
df['Weight'] = df['Weight'].astype(float).round(2)
df.loc[df['Type2'] == '', 'Type2'] = 'None'

df.to_csv('dex.csv', index=False)

In [ ]:
def generate_scores(self):
        '''Calculates distance scores for each available Pokemon and returns sorted dex'''
        if len(self.dex) <= 2:
            return self.dex
        
        dex = self.full_dex.copy()
        dex = dex[~(dex['Name'].isin(self.guesses))]
        for col in ['Generation', 'Height', 'Weight']:
            dex['_' + col] = 0 if col in self.green else dex[col] / self.dex[col].median()

        included_types = [t for t in ['Type1', 'Type2'] if t not in self.green]
        if len(included_types) == 2:
            all_types = pd.concat([self.dex[t] for t in included_types])
            type_scores = all_types.value_counts(True).reindex(dex['Type2'].unique(), fill_value=0)
        elif len(included_types):
            all_types = self.dex[included_types[0]]
            type_scores = all_types.value_counts(True).reindex(dex['Type2'].unique(), fill_value=0)
        else:
            type_scores = None
            dex['type_score'] = 1

        if type_scores is not None:
            dex['type_score'] = (type_scores.loc[dex['Type1']] + type_scores.loc[dex['Type2']].values).values
            dex['type_score'] /= dex['type_score'].max()

        dist = dex[['_Generation', '_Height', '_Weight', 'type_score']].values - np.array([1, 1, 1, 1])
        dex['_distance'] = np.linalg.norm(dist, axis=1)
        dex.sort_values('_distance', inplace=True)
        return dex

In [ ]:
def generate_scores(self):
        '''Calculates distance scores for each available Pokemon and returns sorted dex'''
        dex = self.dex.copy()
        for col in ['Generation', 'Height', 'Weight']:
            dex['_' + col] = dex[col] / dex[col].median()

        all_types = pd.concat([dex['Type1'], dex['Type2']])
        type_scores = all_types.value_counts(True)
        dex['type_score'] = (type_scores.loc[dex['Type1']] + type_scores.loc[dex['Type2']].values).values
        dex['type_score'] /= dex['type_score'].max()

        dist = dex[['_Generation', '_Height', '_Weight', 'type_score']].values - np.array([1, 1, 1, 1])
        dex['_distance'] = np.linalg.norm(dist, axis=1)
        dex.sort_values('_distance', inplace=True)
        return dex